In [ ]:
from momentfm import MOMENTPipeline
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# from tslearn.datasets import UCR_UEA_datasets
# from sklearn.preprocessing import LabelEncoder

In [ ]:
def load_data(file_path):
    data = np.load(file_path, allow_pickle=True).item()
    X_train_full = data["train"]["X"]
    y_train_full = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42
    )
    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = load_data("MP8_projected.npy")

In [ ]:
import time
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


class TimeSeriesDataset(Dataset):
    """
    Custom Dataset class for time series data.
    """

    def __init__(self, data, labels, device):
        self.data = torch.FloatTensor(data).to(device)
        self.labels = torch.LongTensor(labels).to(device)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


def prepare_data_for_moment(X, y):
    """
    Prepare your data for MOMENT model.
    """
    if hasattr(X, "values"):
        X = X.values
    if hasattr(y, "values"):
        y = y.values

    if len(X.shape) == 2:
        X = X.reshape(X.shape[0], 1, -1)

    return X, y


def calculate_accuracy(model, dataloader, device):
    """
    Calculate accuracy for a given dataloader
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, labels in dataloader:
            outputs = model(x_enc=data)
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total


def train_moment_model(
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size=8,
    num_epochs=10,
    learning_rate=1e-4,
    n_channels=3,
    num_classes=3,
    device="mps",
    patience=100,  # Number of epochs to wait before early stopping
):
    """
    Train the MOMENT model with timing, validation accuracy, and early stopping.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Print input shapes and class information
    print(f"\nData Shapes:")
    print(f"Original X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"Number of unique classes in y_train: {len(np.unique(y_train))}")
    print(f"Classes present in y_train: {np.unique(y_train)}")
    print(f"Configured num_classes: {num_classes}\n")

    # Prepare data
    X_train_processed, y_train_processed = prepare_data_for_moment(X_train, y_train)
    print(f"Processed X_train shape: {X_train_processed.shape}")

    # Create dataset and dataloader
    train_dataset = TimeSeriesDataset(X_train_processed, y_train_processed, device)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Prepare validation data
    X_val_processed, y_val_processed = prepare_data_for_moment(X_val, y_val)
    val_dataset = TimeSeriesDataset(X_val_processed, y_val_processed, device)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

    # Initialize model
    model = MOMENTPipeline.from_pretrained(
        "AutonLab/MOMENT-1-small",
        model_kwargs={
            "task_name": "classification",
            "n_channels": n_channels,
            "num_class": num_classes,
        },
    )
    model.init()
    model = model.to(device)

    # Define loss and optimizer
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Early stopping variables
    best_val_accuracy = 0
    epochs_without_improvement = 0
    best_model_state = None

    # Training loop
    start_time = time.time()
    try:
        for epoch in range(num_epochs):
            epoch_start_time = time.time()
            model.train()
            total_loss = 0

            for batch_idx, (data, labels) in enumerate(train_dataloader):
                if epoch == 0 and batch_idx == 0:
                    print("\nFirst batch shapes:")
                    print(f"Input batch shape: {data.shape}")
                    print(f"Labels batch shape: {labels.shape}")

                # Forward pass
                output = model(x_enc=data)

                if epoch == 0 and batch_idx == 0:
                    print(f"Model output logits shape: {output.logits.shape}\n")

                loss = criterion(output.logits, labels)

                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

                if batch_idx % 10 == 0:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - Batch {batch_idx}/{len(train_dataloader)} - "
                        f"Loss: {loss.item():.3f}"
                    )

            # Calculate average loss and validation accuracy
            avg_loss = total_loss / len(train_dataloader)
            val_accuracy = calculate_accuracy(model, val_dataloader, device)
            epoch_time = time.time() - epoch_start_time

            print(
                f"Epoch {epoch + 1}/{num_epochs} - "
                f"Average Loss: {avg_loss:.3f} - "
                f"Validation Accuracy: {val_accuracy:.3f} - "
                f"Time: {epoch_time:.2f}s"
            )

            # Early stopping check
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                epochs_without_improvement = 0
                best_model_state = model.state_dict().copy()
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    print(f"\nEarly stopping triggered after {epoch + 1} epochs")
                    print(f"Best validation accuracy: {best_val_accuracy:.3f}")
                    print(f"\nEarly stopping triggered after {epoch + 1} epochs")
                    break

    except KeyboardInterrupt:
        print("\nTraining interrupted by user")

    print(f"\nEarly stopping triggered after {epoch + 1} epochs")

    total_time = time.time() - start_time
    print(f"\nTotal training time: {total_time:.2f} seconds")

    # Load best model if early stopping was triggered
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model state from early stopping")

    return model, best_val_accuracy, total_time

In [ ]:
# Set device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X_train.shape

In [ ]:
model = train_moment_model(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=16,
    num_epochs=100,
    learning_rate=1e-4,
    n_channels=X_train.shape[1],
    num_classes=len(np.unique(y_train)),
    device="mps",
)

In [ ]:
def predict(model, data, device=None):
    """
    Make predictions using the trained MOMENT model.

    Args:
        model: Trained MOMENT model
        data: Input data of shape (n_samples, n_channels, sequence_length)
        device: torch device (optional)

    Returns:
        predictions: Predicted classes (numpy array)
        probabilities: Class probabilities (numpy array)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model[0].to(device)
    model.eval()

    # Convert to tensor if not already
    if not isinstance(data, torch.Tensor):
        data_tensor = torch.FloatTensor(data)
    else:
        data_tensor = data

    # Move to device
    data_tensor = data_tensor.to(device)

    try:
        with torch.no_grad():
            # Use named argument x_enc for the forward pass
            outputs = model(x_enc=data_tensor)

            # Handle potential output formats
            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            probabilities = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(outputs, dim=1)

            return predictions.cpu().numpy(), probabilities.cpu().numpy()

    except Exception as e:
        print(f"Error during prediction: {str(e)}")
        print(f"Input tensor shape: {data_tensor.shape}")
        raise

In [ ]:
# Make predictions
start_time = time.time()
predictions, probabilities = predict(model, X_test, device="cpu")
total_time = time.time() - start_time
# Get performance metrics
from sklearn.metrics import classification_report, accuracy_score

print("\nClassification Report:")
print(classification_report(y_test, predictions))
print("\nAccuracy Report:")
print(accuracy_score(y_test, predictions))
print(f"total prediction time", total_time)